In [6]:
# =========================================
# SMC Parameter Table Generator (S = N * v * tau)
# =========================================
# Goal: 2D scan over N and tau (like before), with v fixed at a few values.
# This gives: same N but different tau, and same tau but different N, for each v.

import pandas as pd
import itertools

# --- Define parameter axes (~50 total runs: len(N)*len(v)*len(tau) ≈ 50) ---
v_values = [200, 350, 500]              # translocation speeds (3 values)
N_values = [20, 25, 30, 35]             # number of SMCs (4 values)
tau_values = [45, 65, 85, 105]          # dwell times in seconds; grid centered ~700k (4 values)
# Total: 4 × 3 × 4 = 48 runs. For v=350, N*tau ≈ 2000 → S ≈ 700k.

# Optional: restrict to runs where S is in this band (set to None to keep all)
S_transition_min = 5.5e5   # transition ~700k; band 550k–850k
S_transition_max = 8.5e5
filter_by_S_band = False  # set True to keep only runs in [S_transition_min, S_transition_max]

# --- Generate all (N, v, tau) combinations ---
# Full 2D grid over N and tau for each v → guarantees same-N/different-tau and same-tau/different-N
all_runs = list(itertools.product(N_values, v_values, tau_values))

# --- Build table with S = N * v * tau ---
runs_list = []
for run_id, (N, v, tau) in enumerate(all_runs, 1):
    S = N * v * tau
    if filter_by_S_band and (S < S_transition_min or S > S_transition_max):
        continue
    runs_list.append({"run": run_id, "N": N, "v": v, "tau": tau, "S": S})

df = pd.DataFrame(runs_list)

# --- Display settings ---
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

print(f"Total runs: {len(df)} (grid: {len(N_values)} N × {len(v_values)} v × {len(tau_values)} tau)")
if filter_by_S_band:
    print(f"Filtered to S in [{S_transition_min:.0e}, {S_transition_max:.0e}]")
df

Total runs: 48 (grid: 4 N × 3 v × 4 tau)


,run,N,v,tau,S
0,1,20,200,45,180000
1,2,20,200,65,260000
2,3,20,200,85,340000
3,4,20,200,105,420000
4,5,20,350,45,315000
5,6,20,350,65,455000
6,7,20,350,85,595000
7,8,20,350,105,735000
8,9,20,500,45,450000
9,10,20,500,65,650000


In [7]:
#df.sort_values("run")   # by run
df.sort_values("N")     # by N
#df.sort_values("tau")   # by tau
#df.sort_values("S")     # by S

,run,N,v,tau,S
0,1,20,200,45,180000
1,2,20,200,65,260000
2,3,20,200,85,340000
3,4,20,200,105,420000
4,5,20,350,45,315000
5,6,20,350,65,455000
6,7,20,350,85,595000
7,8,20,350,105,735000
8,9,20,500,45,450000
9,10,20,500,65,650000


In [8]:
df.sort_values("tau")   # by tau

,run,N,v,tau,S
0,1,20,200,45,180000
4,5,20,350,45,315000
12,13,25,200,45,225000
8,9,20,500,45,450000
24,25,30,200,45,270000
28,29,30,350,45,472500
20,21,25,500,45,562500
16,17,25,350,45,393750
40,41,35,350,45,551250
44,45,35,500,45,787500


In [9]:
df.sort_values("S")     # by S

,run,N,v,tau,S
0,1,20,200,45,180000
12,13,25,200,45,225000
1,2,20,200,65,260000
24,25,30,200,45,270000
4,5,20,350,45,315000
36,37,35,200,45,315000
13,14,25,200,65,325000
2,3,20,200,85,340000
25,26,30,200,65,390000
16,17,25,350,45,393750


In [1]:
# --- 3D sweep runs table (matches analysis/runs.ipynb and submit_jobs_3d_sweep.sh) ---
import pandas as pd
import itertools

# Parameter grid: must match runs.ipynb and submit_jobs_3d_sweep.sh
N_VALUES = [20, 35, 50]
V_VALUES = [200, 350, 500]
TAU_VALUES = [50, 75, 100]
# Run name prefix for 3D sweep (e.g. Feb16 for submit_jobs_3d_sweep)
RUN_DATE = "Mar03"

all_runs = list(itertools.product(N_VALUES, V_VALUES, TAU_VALUES))
runs_list = []
for run_id, (N, v, tau) in enumerate(all_runs, 1):
    S = N * v * tau
    run_name = f"{RUN_DATE}_p{run_id}"
    runs_list.append({"run": run_id, "run_name": run_name, "N": N, "v": v, "tau": tau, "S": S})

runs_df = pd.DataFrame(runs_list)

def filter_runs(runs_df, run_ids=None, N=None, v=None, tau=None):
    """
    Return subset of runs by run ID(s), N, v, and/or tau.
    E.g. filter_runs(runs_df, N=20) for all runs with N=20.
    """
    out = runs_df.copy()
    if run_ids is not None:
        run_ids = [run_ids] if np.isscalar(run_ids) else list(run_ids)
        out = out[out["run"].isin(run_ids)]
    if N is not None:
        out = out[out["N"] == N]
    if v is not None:
        out = out[out["v"] == v]
    if tau is not None:
        out = out[out["tau"] == tau]
    return out.reset_index(drop=True)

# Example: subset with N=20
# filter_runs(runs_df, N=20)
# filter_runs(runs_df, run_ids=[1, 2, 3])
# filter_runs(runs_df, v=350)
runs_df

,run,run_name,N,v,tau,S
0,1,Mar03_p1,20,200,50,200000
1,2,Mar03_p2,20,200,75,300000
2,3,Mar03_p3,20,200,100,400000
3,4,Mar03_p4,20,350,50,350000
4,5,Mar03_p5,20,350,75,525000
5,6,Mar03_p6,20,350,100,700000
6,7,Mar03_p7,20,500,50,500000
7,8,Mar03_p8,20,500,75,750000
8,9,Mar03_p9,20,500,100,1000000
9,10,Mar03_p10,35,200,50,350000
